In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Joint Refinement: Si, Bragg + PDF

This example demonstrates a joint refinement of the Si crystal
structure combining Bragg diffraction and pair distribution function
(PDF) analysis. The Bragg experiment uses time-of-flight neutron
powder diffraction data from SEPD at Argonne, while the PDF
experiment uses data from NOMAD at SNS. A single shared Si structure
is refined simultaneously against both datasets.

## 🛠️ Import Library

In [2]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data

## 🧩 Define Structure

A single Si structure is shared between the Bragg and PDF
experiments. Structural parameters refined against both datasets
simultaneously.

### Create Structure

In [3]:
structure = StructureFactory.from_scratch(name='si')

### Set Space Group

In [4]:
structure.space_group.name_h_m = 'F d -3 m'
structure.space_group.coord_system_code = '1'

### Set Unit Cell

In [5]:
structure.cell.length_a = 5.42

### Set Atom Sites

In [6]:
structure.atom_sites.create(
    id='Si',
    type_symbol='Si',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    adp_iso=0.2,
)

## 🔬 Define Experiments

Two experiments are defined: one for Bragg diffraction and one for
PDF analysis. Both are linked to the same Si structure.

### Experiment 1: Bragg (SEPD, TOF)

#### Download Data

In [7]:
bragg_data_path = download_data('meas-si-sepd', destination='data')

Getting data...


Data 'meas-si-sepd': Si, SEPD (Argonne)


✅ Data 'meas-si-sepd' already present at '../../../data/meas-si-sepd.xye'. Keeping existing.


#### Create Experiment

In [8]:
bragg_expt = ExperimentFactory.from_data_path(
    name='sepd', data_path=bragg_data_path, beam_mode='time-of-flight'
)

#### Set Instrument

In [9]:
bragg_expt.instrument.setup_twotheta_bank = 144.845
bragg_expt.instrument.calib_d_to_tof_offset = -9.2
bragg_expt.instrument.calib_d_to_tof_linear = 7476.91
bragg_expt.instrument.calib_d_to_tof_quadratic = -1.54

#### Set Peak Profile

In [10]:
bragg_expt.peak.type = 'jorgensen'
bragg_expt.peak.broad_gauss_sigma_0 = 5.0
bragg_expt.peak.broad_gauss_sigma_1 = 45.0
bragg_expt.peak.broad_gauss_sigma_2 = 1.0
bragg_expt.peak.decay_beta_0 = 0.04221
bragg_expt.peak.decay_beta_1 = 0.00946
bragg_expt.peak.rise_alpha_0 = 0.0
bragg_expt.peak.rise_alpha_1 = 0.5971

Peak profile type for experiment 'sepd' changed to


jorgensen


#### Set Background

In [11]:
bragg_expt.background.type = 'line-segment'
for x in range(0, 35000, 5000):
    bragg_expt.background.create(id=str(x), position=x, intensity=200)

Background type for experiment 'sepd' already set to


line-segment


#### Set Linked Structures

In [12]:
bragg_expt.linked_structures.create(structure_id='si', scale=13.0)

### Experiment 2: PDF (NOMAD, TOF)

#### Download Data

In [13]:
pdf_data_path = download_data('meas-si-pdf-nomad', destination='data')

Getting data...


Data 'meas-si-pdf-nomad': Si, NOMAD (SNS), PDF


✅ Data 'meas-si-pdf-nomad' already present at '../../../data/meas-si-pdf-nomad.gr'. Keeping existing.


#### Create Experiment

In [14]:
pdf_expt = ExperimentFactory.from_data_path(
    name='nomad',
    data_path=pdf_data_path,
    beam_mode='time-of-flight',
    scattering_type='total',
)

⚠️ No uncertainty (sy) column provided. Defaulting to 0.03.                                                                       


#### Set Peak Profile (PDF Parameters)

In [15]:
pdf_expt.peak.damp_q = 0.02
pdf_expt.peak.broad_q = 0.02
pdf_expt.peak.cutoff_q = 35.0
pdf_expt.peak.sharp_delta_1 = 0.001
pdf_expt.peak.sharp_delta_2 = 4.0
pdf_expt.peak.damp_particle_diameter = 0

#### Set Linked Structures

In [16]:
pdf_expt.linked_structures.create(structure_id='si', scale=1.0)

## 📦 Define Project

The project object manages the shared structure, both experiments,
and the analysis.

### Create Project

In [17]:
project = Project(name='si_bragg_pdf')

### Add Structure

In [18]:
project.structures.add(structure)

### Add Experiments

In [19]:
project.experiments.add(bragg_expt)
project.experiments.add(pdf_expt)

## 🚀 Perform Analysis

This section shows the joint analysis process. The calculator is
auto-resolved per experiment: CrysPy for Bragg, PDFfit for PDF.

### Set Fit Mode and Weights

In [20]:
project.analysis.fitting_mode.type = 'joint'
project.analysis.joint_fit.create(experiment_id='sepd', weight=0.7)
project.analysis.joint_fit.create(experiment_id='nomad', weight=0.3)

Fitting mode changed to


joint


### Display Structure

In [21]:
project.display.structure(struct_name='si')

Structure 🧩 'si' (Atom view type: 'covalent')


### Display Pattern (Before Fit)

In [22]:
project.display.pattern(expt_name='sepd')

In [23]:
project.display.pattern(expt_name='nomad')

### Set Free Parameters

Shared structural parameters are refined against both datasets
simultaneously.

In [24]:
structure.cell.length_a.free = True
structure.atom_sites['Si'].adp_iso.free = True

Bragg experiment parameters.

In [25]:
bragg_expt.linked_structures['si'].scale.free = True
bragg_expt.instrument.calib_d_to_tof_offset.free = True
bragg_expt.peak.broad_gauss_sigma_0.free = True
bragg_expt.peak.broad_gauss_sigma_1.free = True
bragg_expt.peak.broad_gauss_sigma_2.free = True
for point in bragg_expt.background:
    point.intensity.free = True

PDF experiment parameters.

In [26]:
pdf_expt.linked_structures['si'].scale.free = True
pdf_expt.peak.damp_q.free = True
pdf_expt.peak.broad_q.free = True
pdf_expt.peak.sharp_delta_1.free = True
pdf_expt.peak.sharp_delta_2.free = True

### Display Free Parameters

In [27]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.42000,,-inf,inf,Å
2,si,atom_site,Si,adp_iso,0.20000,,-inf,inf,Å²
3,sepd,linked_structure,si,scale,13.00000,,-inf,inf,
4,sepd,peak,,broad_gauss_sigma_0,5.00000,,-inf,inf,μs²
5,sepd,peak,,broad_gauss_sigma_1,45.00000,,-inf,inf,μs/Å
6,sepd,peak,,broad_gauss_sigma_2,1.00000,,-inf,inf,μs²/Å²
7,sepd,instrument,,d_to_tof_offset,-9.20000,,-inf,inf,μs
8,sepd,background,0,intensity,200.00000,,-inf,inf,
9,sepd,background,5000,intensity,200.00000,,-inf,inf,
10,sepd,background,10000,intensity,200.00000,,-inf,inf,


### Run Fitting

In [28]:
project.analysis.fit()
project.display.fit.results()
project.display.fit.correlations()

<IPython.core.display.Javascript object>

Using all experiments 🔬 ['sepd', 'nomad'] for 'joint' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.54,3092.49,
2,10,5.57,3092.49,
3,20,11.04,3092.46,
4,23,12.68,1088.01,64.8% ↓
5,31,18.10,1088.01,
6,39,23.69,1088.02,
7,43,26.34,494.69,54.5% ↓
8,50,31.81,494.69,
9,57,37.16,494.69,
10,63,42.96,193.42,60.9% ↓


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),262.35
4,🔁 Iterations,372
5,📏 Goodness-of-fit (reduced χ²),52.16
6,"📏 R-factor (Rf, %)",10.61
7,"📏 R-factor squared (Rf², %)",9.69
8,"📏 Weighted R-factor (wR, %)",9.26


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4200,5.4306,0.0000,0.20 % ↑
2,si,atom_site,Si,adp_iso,Å²,0.2000,0.7080,0.0037,254.01 % ↑
3,sepd,linked_structure,si,scale,,13.0000,669.5144,4.2204,5050.11 % ↑
4,sepd,peak,,broad_gauss_sigma_0,μs²,5.0000,-9.9687,0.4430,299.37 % ↓
5,sepd,peak,,broad_gauss_sigma_1,μs/Å,45.0000,61.0087,2.1638,35.57 % ↑
6,sepd,peak,,broad_gauss_sigma_2,μs²/Å²,1.0000,-1.2086,0.4620,220.86 % ↓
7,sepd,instrument,,d_to_tof_offset,μs,-9.2000,-8.2414,0.0867,10.42 % ↓
8,sepd,background,0,intensity,,200.0000,280.9113,3.2399,40.46 % ↑
9,sepd,background,5000,intensity,,200.0000,149.3418,1.3519,25.33 % ↓
10,sepd,background,10000,intensity,,200.0000,118.0461,1.4229,40.98 % ↓


### Display Pattern (After Fit)

In [29]:
project.display.pattern(expt_name='sepd')

In [30]:
project.display.pattern(expt_name='nomad')

## 💾 Save Project

In [31]:
project.save_as(dir_path='projects/joint-si-bragg-pdf')

Saving project 📦 'si_bragg_pdf' to '../../../projects/joint-si-bragg-pdf'


├── 📄 project.edi


├── 📁 structures/


│   └── 📄 si.edi


├── 📁 experiments/


...

│   └── 📄 nomad.edi


├── 📁 analysis/


│   └── 📄 analysis.edi


└── 📁 reports/


    └── 📄 si_bragg_pdf.html
